# Industrial Water Treatment Telemetry & Anomaly Pipeline (DIKW)

```
[Simulator] -> [Raw warehouse table] -> [dbt star schema + tests] -> [RandomForest] -> [Alerts & dashboard]
   DATA            INFORMATION               KNOWLEDGE                  WISDOM            WISDOM
```

**Colab adaptations (so everything runs in one notebook, free):**
- **DuckDB** stands in for GCP BigQuery (same SQL/dbt workflow; swap the dbt adapter to `dbt-bigquery` for production).
- **Airflow** DAG is written to `airflow_dag.py` for reference (Docker/Airflow can't run inside Colab); the notebook runs the same batch-load logic in a loop.
- Dashboard is rendered inline with matplotlib, plus a ready-to-run `app.py` for Streamlit.

## 0. Setup

In [ ]:
!pip -q install duckdb dbt-duckdb scikit-learn matplotlib pandas numpy

In [ ]:
import numpy as np, pandas as pd, duckdb, pathlib, textwrap, subprocess
import matplotlib.pyplot as plt

np.random.seed(42)
WORK = pathlib.Path("water_dikw"); WORK.mkdir(exist_ok=True)
DB = str((WORK / "warehouse.duckdb").resolve())

## 1. DATA layer: simulate telemetry from 50 plants
5 sensors per plant, one reading every 15 min for 3 days. Two anomaly types are injected:
- **pump_failure**: pressure slowly drifts down for 2 hours (8 steps), then collapses (predictable)
- **ph_spike**: sudden pH jump (not predictable, caught by a rule)

In [ ]:
N_PLANTS, STEPS = 50, 288
ts = pd.date_range("2026-01-01", periods=STEPS, freq="15min")
t_idx = np.arange(STEPS)
UNITS = {"ph": "pH", "turbidity": "NTU", "flow": "m3/h", "pressure": "bar", "chem_dose": "mg/L"}

facilities = pd.DataFrame({
    "facility_id": [f"WTP-{i:03d}" for i in range(1, N_PLANTS + 1)],
    "facility_name": [f"Plant {i}" for i in range(1, N_PLANTS + 1)],
    "region": np.random.choice(["North", "South", "East", "West"], N_PLANTS),
    "capacity_m3_h": np.random.choice([300, 500, 800, 1200], N_PLANTS),
})

rows, events = [], []
for fid, cap in zip(facilities.facility_id, facilities.capacity_m3_h):
    d = {
        "ph": np.random.normal(7.2, 0.08, STEPS),
        "turbidity": np.abs(np.random.normal(0.8, 0.15, STEPS)),
        "flow": cap * 0.6 * (1 + 0.1 * np.sin(2 * np.pi * t_idx / 96)) + np.random.normal(0, cap * 0.01, STEPS),
        "pressure": np.random.normal(4.0, 0.08, STEPS),
        "chem_dose": np.random.normal(2.0, 0.1, STEPS),
    }
    for _ in range(np.random.randint(1, 3)):          # pump failures with 2h precursor
        t = np.random.randint(20, STEPS - 8)
        for i in range(t - 8, t):
            frac = (i - (t - 8) + 1) / 8
            d["pressure"][i] -= 1.0 * frac
            d["flow"][i] *= 1 - 0.08 * frac
        d["pressure"][t:t + 4] -= 2.5
        d["flow"][t:t + 4] *= 0.4
        events.append((fid, ts[t], "pump_failure"))
    for t in np.random.choice(STEPS, 3, replace=False):  # sudden pH spikes
        d["ph"][t] += np.random.choice([-2.0, 2.5])
        events.append((fid, ts[t], "ph_spike"))
    for s, v in d.items():
        rows.append(pd.DataFrame({"facility_id": fid, "sensor_id": f"{fid}-{s}", "sensor_type": s,
                                  "unit": UNITS[s], "reading_ts": ts, "value": v.round(3)}))

raw = pd.concat(rows, ignore_index=True)
events = pd.DataFrame(events, columns=["facility_id", "event_ts", "event_type"])
print(raw.shape, "| events:", events.event_type.value_counts().to_dict())
raw.head()

## 2. INFORMATION layer: batch ingestion into the raw (bronze) layer
Loads one "day" per batch, like a scheduled Airflow run would. Reference DAG is saved to `airflow_dag.py`.

In [ ]:
con = duckdb.connect(DB)
con.execute("create or replace table raw_sensor_readings as select *, now() as _loaded_at from raw where 1=0")

for day, batch in raw.groupby(raw.reading_ts.dt.date):      # = one scheduled batch load
    con.execute("insert into raw_sensor_readings select *, now() from batch")
    print("loaded batch", day, len(batch), "rows")

con.execute("create or replace table raw_facilities as select * from facilities")
con.execute("create or replace table raw_events as select * from events")
print(con.execute("select count(*) from raw_sensor_readings").fetchone())
con.close()   # DuckDB allows one writer; release it so dbt can connect

In [ ]:
(WORK / "airflow_dag.py").write_text("""
# Reference Airflow DAG (run in Docker/Cloud Composer). In production, swap DuckDB for BigQuery.
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator

def extract_and_load(**ctx):
    from google.cloud import bigquery
    from simulator import generate_batch          # your simulator module
    df = generate_batch(ctx["data_interval_start"], ctx["data_interval_end"])
    client = bigquery.Client()
    client.load_table_from_dataframe(df, "my_project.bronze.raw_sensor_readings").result()

with DAG("water_telemetry_hourly", start_date=datetime(2026, 1, 1), schedule="@hourly",
         catchup=False, default_args={"retries": 2, "retry_delay": timedelta(minutes=5)}) as dag:
    load = PythonOperator(task_id="load_raw", python_callable=extract_and_load)
    dbt_build = BashOperator(task_id="dbt_build", bash_command="cd /opt/dbt && dbt build")
    load >> dbt_build
""")
print("wrote airflow_dag.py")

## 3. KNOWLEDGE layer: dbt star schema + automated data tests
Creates a dbt project: `stg_readings` (dedupe/clean) → `dim_facility`, `dim_sensor`, `fact_sensor_readings`, with tests (no null sensor IDs, pH between 0 and 14, keys unique, referential integrity).

In [ ]:
DBT = WORK / "dbt_project"
for p in ["models/staging", "models/marts", "tests"]:
    (DBT / p).mkdir(parents=True, exist_ok=True)

(DBT / "dbt_project.yml").write_text(textwrap.dedent("""
    name: water
    version: "1.0"
    profile: water
    model-paths: ["models"]
    test-paths: ["tests"]
    models:
      water:
        staging: {+materialized: view}
        marts: {+materialized: table}
"""))
(DBT / "profiles.yml").write_text(textwrap.dedent(f"""
    water:
      target: dev
      outputs:
        dev:
          type: duckdb
          path: {DB}
          threads: 1
"""))
(DBT / "models/staging/sources.yml").write_text(textwrap.dedent("""
    version: 2
    sources:
      - name: raw
        schema: main
        tables:
          - name: raw_sensor_readings
          - name: raw_facilities
          - name: raw_events
"""))
(DBT / "models/staging/stg_readings.sql").write_text(textwrap.dedent("""
    select
        facility_id, sensor_id, sensor_type, unit,
        cast(reading_ts as timestamp) as reading_ts,
        cast(value as double) as value
    from {{ source('raw', 'raw_sensor_readings') }}
    qualify row_number() over (partition by sensor_id, reading_ts order by _loaded_at desc) = 1
"""))
(DBT / "models/marts/dim_facility.sql").write_text(textwrap.dedent("""
    select row_number() over (order by facility_id) as facility_key,
           facility_id, facility_name, region, capacity_m3_h
    from {{ source('raw', 'raw_facilities') }}
"""))
(DBT / "models/marts/dim_sensor.sql").write_text(textwrap.dedent("""
    select distinct sensor_id, facility_id, sensor_type, unit
    from {{ ref('stg_readings') }}
"""))
(DBT / "models/marts/fact_sensor_readings.sql").write_text(textwrap.dedent("""
    select sensor_id, facility_id, reading_ts,
           cast(reading_ts as date) as date_key, value
    from {{ ref('stg_readings') }}
"""))
(DBT / "models/marts/schema.yml").write_text(textwrap.dedent("""
    version: 2
    models:
      - name: dim_facility
        columns:
          - name: facility_id
            tests: [unique, not_null]
      - name: dim_sensor
        columns:
          - name: sensor_id
            tests: [unique, not_null]
      - name: fact_sensor_readings
        columns:
          - name: sensor_id
            tests:
              - not_null
              - relationships: {to: ref('dim_sensor'), field: sensor_id}
          - name: facility_id
            tests:
              - relationships: {to: ref('dim_facility'), field: facility_id}
          - name: value
            tests: [not_null]
"""))
(DBT / "tests/assert_ph_between_0_and_14.sql").write_text(textwrap.dedent("""
    select * from {{ ref('fact_sensor_readings') }} f
    join {{ ref('dim_sensor') }} s using (sensor_id)
    where s.sensor_type = 'ph' and (f.value < 0 or f.value > 14)
"""))

res = subprocess.run(["dbt", "build", "--profiles-dir", "."], cwd=DBT, capture_output=True, text=True)
print(res.stdout[-3000:], res.stderr[-1500:])
assert res.returncode == 0, "dbt build failed (a model or data test failed)"

## 4. WISDOM layer: predict pump failures 2 hours ahead
Reads the star schema, pivots to one row per plant/timestamp, engineers rolling features, and trains a RandomForest. Label = *a pump failure starts within the next 2 hours*. Split is chronological (train days 1–2, test day 3).

In [ ]:
con = duckdb.connect(DB, read_only=True)
fact = con.execute("""
    select f.facility_id, f.reading_ts, s.sensor_type, f.value
    from fact_sensor_readings f join dim_sensor s using (sensor_id)
""").df()
dim_fac = con.execute("select * from dim_facility").df()
events = con.execute("select * from raw_events").df()
con.close()

wide = (fact.pivot_table(index=["facility_id", "reading_ts"], columns="sensor_type", values="value")
            .reset_index().sort_values(["facility_id", "reading_ts"]).reset_index(drop=True))

# labels
step = pd.Timedelta("15min")
wide["label"], wide["in_failure"] = 0, False
for fid, t in events[events.event_type == "pump_failure"][["facility_id", "event_ts"]].itertuples(index=False):
    m = wide.facility_id == fid
    wide.loc[m & (wide.reading_ts >= t - 8 * step) & (wide.reading_ts < t), "label"] = 1
    wide.loc[m & (wide.reading_ts >= t) & (wide.reading_ts < t + 4 * step), "in_failure"] = True

# features
g = wide.groupby("facility_id")
wide["flow_ratio"] = wide.flow / g.flow.transform("median")
g = wide.groupby("facility_id")
wide["p_diff1"] = g.pressure.diff()
wide["p_diff8"] = g.pressure.diff(8)
wide["p_mean4"] = g.pressure.transform(lambda s: s.rolling(4, min_periods=1).mean())
wide["p_std4"] = g.pressure.transform(lambda s: s.rolling(4, min_periods=2).std())
wide["f_mean4"] = g.flow_ratio.transform(lambda s: s.rolling(4, min_periods=1).mean())
FEATURES = ["ph", "turbidity", "pressure", "chem_dose", "flow_ratio", "p_diff1", "p_diff8", "p_mean4", "p_std4", "f_mean4"]
wide[FEATURES] = wide[FEATURES].fillna(0)

cutoff = ts[192]  # first 2 days = train
train = wide[(wide.reading_ts < cutoff) & ~wide.in_failure]
test = wide[(wide.reading_ts >= cutoff) & ~wide.in_failure].copy()

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(train[FEATURES], train.label)
test["risk"] = rf.predict_proba(test[FEATURES])[:, 1]
print("ROC-AUC:", round(roc_auc_score(test.label, test.risk), 3))
print(classification_report(test.label, test.risk >= 0.5, digits=3))

## 5. Alerts, dispatch recommendations & dashboard

In [ ]:
def recommend(r):
    if r >= 0.8: return "P1: dispatch engineer now - inspect pump & pressure line"
    if r >= 0.5: return "P2: schedule inspection within 2 hours"
    return "OK"

test = test.merge(dim_fac[["facility_id", "region"]], on="facility_id")
alerts = (test[test.risk >= 0.5].groupby(["facility_id", "region"])
          .agg(first_alert=("reading_ts", "min"), alert_rows=("risk", "size"), max_risk=("risk", "max"))
          .reset_index().sort_values("max_risk", ascending=False))
alerts["recommendation"] = alerts.max_risk.map(recommend)

latest = test.sort_values("reading_ts").groupby("facility_id").tail(1)
health = latest[["facility_id", "region", "reading_ts", "pressure", "ph", "risk"]].sort_values("risk", ascending=False)
ph_alerts = test[(test.ph < 6.5) | (test.ph > 8.5)]

alerts.to_csv(WORK / "alerts.csv", index=False)
health.to_csv(WORK / "health.csv", index=False)
print(f"Plants alerting: {len(alerts)} | pH rule alerts: {len(ph_alerts)}")
alerts.head(10)

In [ ]:
top = alerts.facility_id.iloc[0]
d = test[test.facility_id == top]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))

alerts.head(10).set_index("facility_id").max_risk.plot.barh(ax=ax[0], color="crimson")
ax[0].invert_yaxis(); ax[0].set_title("Top 10 plants by failure risk")

ax[1].plot(d.reading_ts, d.pressure, label="pressure (bar)")
ax[1].plot(d.reading_ts, d.risk * 4, "r--", label="risk x4")
for _, r in events[(events.facility_id == top) & (events.event_type == "pump_failure")].iterrows():
    ax[1].axvline(r.event_ts, color="k", ls=":")
ax[1].set_title(f"{top}: pressure vs predicted risk (dotted = failure)"); ax[1].legend()

health.risk.plot.hist(bins=20, ax=ax[2]); ax[2].set_title("Current risk across all plants")
plt.tight_layout(); plt.show()

In [ ]:
(WORK / "app.py").write_text("""
import pandas as pd, streamlit as st
st.set_page_config(page_title="Water Plant Health", layout="wide")
st.title("Water Treatment Network - Health & Alerts")
alerts = pd.read_csv("alerts.csv"); health = pd.read_csv("health.csv")
c1, c2, c3 = st.columns(3)
c1.metric("Plants monitored", len(health))
c2.metric("Plants with alerts", len(alerts))
c3.metric("P1 (urgent)", int((alerts.max_risk >= 0.8).sum()))
st.subheader("Alerts & dispatch recommendations")
st.dataframe(alerts, use_container_width=True)
st.subheader("Current plant risk")
st.bar_chart(health.set_index("facility_id").risk)
""")
print("Streamlit app saved. To run it (outside Colab, or via a tunnel): cd water_dikw && streamlit run app.py")

## Pitch
> I built an end-to-end industrial water telemetry pipeline on the DIKW model. I ingested synthetic sensor streams into a warehouse raw layer, modelled them into a Kimball star schema with dbt and automated data-quality tests, and trained a RandomForest to flag pump failures two hours ahead, driving prioritised maintenance dispatch alerts.

**Production swap:** `dbt-duckdb` → `dbt-bigquery`, loop → the Airflow DAG in `airflow_dag.py`, matplotlib → Streamlit/Looker Studio.